# Wall Segmentation

**Stage 6**  - Segment each room's wall cloud into individual planar walls and flatten them into 2D images.

**Input:** Room wall clouds (`.ply`) from any upstream wall-assignment stage (geometric, SAM, or geometric+SAM).

**Output:** Per-room wall images (black = wall surface, white = void) + `wall_meta.json` with wall geometry.

The wall images are consumed by the next notebook (door/window detection).

In [ ]:
import sys, os, glob, logging

def _find_root():
    d = os.path.abspath('')
    while True:
        if os.path.isfile(os.path.join(d, 'pyproject.toml')):
            return d
        p = os.path.dirname(d)
        if p == d:
            return os.path.abspath('')
        d = p

PROJECT_ROOT = _find_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
sys.path = [p for p in sys.path if not p.endswith('/src') and not p.endswith('\\src')]
for mod in list(sys.modules):
    if mod == 'scan2bim' or mod.startswith('scan2bim.'):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format='%(message)s')
print('Project root:', PROJECT_ROOT)

In [ ]:
import scan2bim
from scan2bim import artifacts as A

CFG = scan2bim.load_config(start=PROJECT_ROOT)
print(f'Input cloud: {CFG.file_path}')
print(f'Output root: {CFG.out_root}')
print(f'Wall image resolution: {CFG.flat_pixel_m} m/px')

## Select upstream wall stage

Choose which method's wall clouds to process. Change `WALL_STAGE` to match your upstream run.

In [ ]:
# Choose one:
WALL_STAGE = A.STAGE3          # geometric method
# WALL_STAGE = A.STAGE_SAM_WALLS  # pure-SAM method
# WALL_STAGE = A.STAGE5          # geometric + SAM method

wall_dir = A.load_stage_dir(CFG.out_root, WALL_STAGE)
room_clouds = sorted(glob.glob(os.path.join(wall_dir, 'room_*_walls.ply')))
print(f'Found {len(room_clouds)} room wall clouds in {WALL_STAGE}')
for p in room_clouds:
    print(f'  {os.path.basename(p)}')

## Run wall segmentation

In [ ]:
from scan2bim.wall_seg import run_wall_segmentation

out_dir = A.ensure_dir(A.stage_dir(CFG.out_root, A.STAGE_WALL_SEG))
results = run_wall_segmentation(room_clouds, CFG, out_dir)

total = sum(len(v) for v in results.values())
print(f'\nTotal: {len(results)} rooms, {total} wall images')

## Visualize wall images

In [ ]:
from scan2bim.viz import show_wall_images

for room_name, flats in results.items():
    show_wall_images(flats, room_name=room_name)

## Save config + package

In [ ]:
A.save_config(os.path.join(out_dir, A.CONFIG_JSON), CFG)
z = A.package_stage(CFG.out_root, A.STAGE_WALL_SEG)
print(f'Packaged: {z}')